In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd
import os
from scipy.ndimage import center_of_mass
from tqdm.notebook import tqdm # Important for Jupyter widgets
import matplotlib.pyplot as plt

In [ ]:
def process_all_patients(folder_path):
    all_data = []
    
    # 1. Get the list of NIfTI files
    files = [f for f in os.listdir(folder_path) if f.endswith(".nii.gz") and not f.startswith("._")]
    
    # 2. Setup the MASTER Progress Bar (Overall)
    master_pbar = tqdm(files, desc="Overall Progress", unit="patient")
    
    for filename in master_pbar:
        patient_id = filename.split('_')[0]
        file_path = os.path.join(folder_path, filename)
        
        try:
            # Load the 3D volume
            img = nib.load(file_path)
            data = img.get_fdata()
            affine = img.affine
            
            # Find the labels (electrodes)
            unique_labels = np.unique(data)
            unique_labels = unique_labels[unique_labels != 0]
            
            # 3. Setup the PATIENT Progress Bar
            # leave=False ensures this bar vanishes after the patient is done
            patient_pbar = tqdm(unique_labels, 
                                desc=f"└─ Patient {patient_id}", 
                                leave=False, 
                                unit="lead")
            
            for label in patient_pbar:
                mask = (data == label)
                pixel_coords = center_of_mass(mask)
                world_coords = nib.affines.apply_affine(affine, pixel_coords)
                
                all_data.append({
                    'Patient_ID': patient_id,
                    'Label': int(label),
                    'X_mm': world_coords[0],
                    'Y_mm': world_coords[1],
                    'Z_mm': world_coords[2]
                })
                
        except Exception as e:
            print(f"⚠️ Skipping {patient_id} due to error: {e}")
            
    print("✨ All extraction tasks completed!")
    return pd.DataFrame(all_data)

In [ ]:
# --- CONFIGURATION ---
data_folder = r"C:\Users\BENG 280C Project\BENG280C_pacing_lead_data_seg_nii_1st20\HCT2_leads_seg_nii"

if os.path.exists(data_folder):
    df_results = process_all_patients(data_folder)
    # This renders a clean table in your notebook
    display(df_results.head(10)) 
    df_results.to_csv('batch_electrode_coordinates.csv', index=False)
else:
    print(f"❌ Could not find folder: {data_folder}")

Overall Progress:   0%|          | 0/17 [00:00<?, ?patient/s]

└─ Patient 10001:   0%|          | 0/9 [00:00<?, ?lead/s]

└─ Patient 10002:   0%|          | 0/9 [00:00<?, ?lead/s]

└─ Patient 10003:   0%|          | 0/9 [00:00<?, ?lead/s]

└─ Patient 10004:   0%|          | 0/9 [00:00<?, ?lead/s]

...

└─ Patient 10018:   0%|          | 0/9 [00:00<?, ?lead/s]

└─ Patient 10019:   0%|          | 0/9 [00:00<?, ?lead/s]

└─ Patient 10020:   0%|          | 0/9 [00:00<?, ?lead/s]

✨ All extraction tasks completed!


,Patient_ID,Label,X_mm,Y_mm,Z_mm
0,10001,4001,-75.323242,199.131836,224.899992
1,10001,4002,-96.272461,211.295898,177.999993
2,10001,4003,-48.291992,139.325195,214.399993
3,10001,4004,-100.665039,136.622070,203.899993
4,10001,4005,-89.852539,124.120117,208.799993
5,10001,4006,-81.405273,120.065430,208.099993
6,10001,4007,-69.916992,114.997070,216.499993
7,10001,4008,-77.012695,222.446289,203.899993
8,10001,4009,-67.889648,222.108398,200.399993
9,10002,4001,-35.683594,234.300781,287.199995


In [9]:
print("done")

done
